# Custom Tirex Evaluation

In [1]:
import sys
sys.path.append('../tirex/src') # Add the path to the tirex module

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pandas as pd
import torch
import json
import altair as alt
from datetime import datetime
from pathlib import Path

from typing import Dict, Optional

from tirex import ForecastModel, load_model
from tirex_loss.models.base_model import Base_Model
from tirex_loss.logger import TrainingLogger
from tirex_loss.logger.plot import plot_training_curves

# set default figure size for all plots
plt.rcParams["figure.figsize"] = (12, 6)
# set default seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load data

Download the data from the notebook [data_download.ipynb](data_download.ipynb) before.

## Fine-Tuning Data

In [3]:
data_path_fets = "../data/data_fets.csv"
data_fets = pl.read_csv(data_path_fets, try_parse_dates=True)
data_fets

timestamp,value,series_index
"datetime[μs, UTC]",f64,i64
2019-12-31 23:00:00 UTC,43881.8,0
2019-12-31 23:15:00 UTC,43639.6,0
2019-12-31 23:30:00 UTC,43330.9,0
2019-12-31 23:45:00 UTC,43149.5,0
2020-01-01 00:00:00 UTC,43017.3,0
…,…,…
2025-10-13 20:45:00 UTC,51524.0,0
2025-10-13 21:00:00 UTC,50427.2,0
2025-10-13 21:15:00 UTC,49845.8,0


# Load Model

In [4]:
# use custom model
model = Base_Model(context_length=1024,
                   quantiles=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
                   patch_size=32,
                   use_slstm=True)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model device: {device}")

Total Parameters: 1,610,816
Model device: cuda


In [5]:
# for restoring the original behavior of the model after our modifications
from tirex import TiRexZero

TiRexZero._original_forecast_tensor = TiRexZero._forecast_tensor

def restore_original_behavior(print_message: bool = True):
    if hasattr(TiRexZero, '_original_forecast_tensor'):
        TiRexZero._forecast_tensor = TiRexZero._original_forecast_tensor
        if print_message:
            print("Restored original NaN-filling behavior.")
    else:
        print("Original method backup not found.")

# Dataloader

Dataloader produces sequences of length $n$ for the input and target sequences with random predictions lengths in the range of $s_{\min}$ and $s_{\max}$.

In [6]:
from torch.utils.data import DataLoader
from tirex_loss.dataloader import build_dataloader, build_dataloader_from_dataset, TirexDataset, TirexDataset_Fixed
from tirex_loss.dataloader.utils import create_windows, create_windows_fixed

In [7]:
n = model.context_length
s_min = 32
s_max = 128
batch_size = 1 # use batch size 1, sklearn metrics need input (n_samples,)
train_ratio = 0.8

data_sets_filtered = data_fets
seq, lengths = create_windows_fixed(data_sets_filtered, n=n, s_min=s_min, s_max=s_max)

idx = np.random.permutation(len(seq))
seq, lengths = seq[idx], lengths[idx]

split = int(len(seq) * train_ratio)
seq_train, seq_test = seq[:split], seq[split:]
lengths_train, lengths_test = lengths[:split], lengths[split:]

dataset_train = TirexDataset_Fixed(sequences=seq_train,
                             prediction_lengths=lengths_train,
                             context_length=n,
                             prediction_length_min=s_min,
                             prediction_length_max=s_max,
                             mode='shift'
                             )
dataset_test = TirexDataset_Fixed(sequences=seq_test,
                            prediction_lengths=lengths_test,
                            context_length=n,
                            prediction_length_min=s_min,
                            prediction_length_max=s_max,
                            mode='shift'
                            )

dataloader_train = build_dataloader_from_dataset(dataset_train,
                                    batch_size=batch_size,
                                    shuffle=True,
                                    num_workers=0,
                                    pin_memory=True
                                    )
dataloader_test = build_dataloader_from_dataset(dataset_test,
                                   batch_size=batch_size,
                                   shuffle=False,
                                   num_workers=0,
                                   pin_memory=True
                                   )

len(dataloader_train), len(dataloader_test)

(5043, 1261)

# Evaluation

In [8]:
from tqdm.auto import tqdm
from tirex_loss.modifications.autoregressive import autoregressive_mean_forecast_tensor
from tirex_loss.evaluation.metrics import score_data

model_files = ['best_model.pth', 'last_model.pth']

In [9]:
def compare_autoregressive(dataloader_test: DataLoader, device:str):
    model.eval()
    scores_comparison = {"original": (0, 0, 0, 0, 0), "autoregressive_mean": (0, 0, 0, 0, 0)}
    quantiles_comparison = {"original": [], "autoregressive_mean": []}

    batch_bar = tqdm(dataloader_test, total=len(dataloader_test),
                            leave=False, position=1, desc='Batches')
    for i, (x_batch, y_batch, pred_length_batch) in enumerate(batch_bar):
        # put tensors on same device
        x = x_batch.to(device)
        y = y_batch.to(device)
        pred_length = pred_length_batch[0].item()

        # change shape to (n_samples, n_outputs)
        y_true = y.reshape(-1, 1)
        y_train = x.reshape(-1, 1)

        quantiles_original = model(x,
                                    prediction_length=pred_length,
                                    autoregressive=False
                                    )
        quantiles_comparison["original"].append(quantiles_original.detach().cpu().numpy())

        # scores, output is (batch, quantile, time), need to reshape to (batch*time, 1, quantile) for scoring
        y_pred = quantiles_original.reshape(quantiles_original.shape[0]*quantiles_original.shape[2],1,-1)
        score_mape, score_mase, score_r2, score_wis, score_quantile = score_data(
                    y_true=y_true.detach().cpu().numpy(),
                    y_pred=y_pred.detach().cpu().numpy(),
                    y_train=y_train.detach().cpu().numpy(),
                    quantiles=model.quantiles,
                    log_scores=False,
                    multioutput='uniform_average'
                    )

        scores_comparison["original"] = tuple(a + b for a, b in zip(scores_comparison["original"], (score_mape, score_mase, score_r2, score_wis, score_quantile)))

        # now with autoregressive mean filling
        quantiles_autoregressive = model(x,
                                            prediction_length=pred_length,
                                            autoregressive=True
                                            )
        quantiles_comparison["autoregressive_mean"].append(quantiles_autoregressive.detach().cpu().numpy())

        # scores
        y_pred = quantiles_autoregressive.reshape(quantiles_autoregressive.shape[0]*quantiles_autoregressive.shape[2],1,-1)
        score_mape, score_mase, score_r2, score_wis, score_quantile = score_data(
                    y_true=y_true.detach().cpu().numpy(),
                    y_pred=y_pred.detach().cpu().numpy(),
                    y_train=y_train.detach().cpu().numpy(),
                    quantiles=model.quantiles,
                    log_scores=False,
                    multioutput='uniform_average'
                    )

        scores_comparison["autoregressive_mean"] = tuple(a + b for a, b in zip(scores_comparison["autoregressive_mean"], (score_mape, score_mase, score_r2, score_wis, score_quantile)))

    scores_comparison["original"] = tuple(score / len(dataloader_test) for score in scores_comparison["original"])
    scores_comparison["autoregressive_mean"] = tuple(score / len(dataloader_test) for score in scores_comparison["autoregressive_mean"])    

    return scores_comparison, quantiles_comparison

def save_scores(scores: dict, path: str = "scores.json") -> None:
    with open(path, "w") as f:
        json.dump(scores, f, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else x)

In [10]:
def compare_autoregressive_pretrained(dataloader_test: DataLoader, device:str, run_autoregressive: Optional[bool] = True):
    scores_comparison = {"original": (0, 0, 0, 0, 0), "autoregressive_mean": (0, 0, 0, 0, 0)}
    quantiles_comparison = {"original": [], "autoregressive_mean": []}
    mean_comparison = {"original": [], "autoregressive_mean": []}

    batch_bar = tqdm(dataloader_test, total=len(dataloader_test),
                            leave=False, position=1, desc='Batches')
    for i, (x_batch, y_batch, pred_length_batch) in enumerate(batch_bar):
        # put tensors on same device
        x = x_batch.to(device)
        y = y_batch.to(device)
        pred_length = pred_length_batch[0].item()

        # change shape to (n_samples, n_outputs)
        y_true = y.reshape(-1, 1)
        y_train = x.reshape(-1, 1)

        restore_original_behavior(print_message=False)
        quantiles_original, mean_original = model.forecast(x,
                                                           prediction_length=pred_length,
                                                           output_device=device
                                                           )
        quantiles_comparison["original"].append(quantiles_original.cpu().numpy())
        mean_comparison["original"].append(mean_original.cpu().numpy())

        # scores
        y_pred = quantiles_original.reshape(quantiles_original.shape[0]*quantiles_original.shape[1],1,-1)
        score_mape, score_mase, score_r2, score_wis, score_quantile = score_data(
                    y_true=y_true.cpu().numpy(),
                    y_pred=y_pred.cpu().numpy(),
                    y_train=y_train.cpu().numpy(),
                    quantiles=model.config.quantiles,
                    log_scores=False,
                    multioutput='uniform_average'
                    )

        scores_comparison["original"] = tuple(a + b for a, b in zip(scores_comparison["original"], (score_mape, score_mase, score_r2, score_wis, score_quantile)))

        # now with autoregressive mean filling
        if run_autoregressive:
            TiRexZero._forecast_tensor = autoregressive_mean_forecast_tensor
            quantiles_autoregressive, mean_autoregressive = model.forecast(x,
                                                                        prediction_length=pred_length,
                                                                        output_device=device
                                                                        )
            quantiles_comparison["autoregressive_mean"].append(quantiles_autoregressive.cpu().numpy())
            mean_comparison["autoregressive_mean"].append(mean_autoregressive.cpu().numpy())

            # scores
            y_pred = quantiles_autoregressive.reshape(quantiles_autoregressive.shape[0]*quantiles_autoregressive.shape[1],1,-1)
            score_mape, score_mase, score_r2, score_wis, score_quantile = score_data(
                        y_true=y_true.cpu().numpy(),
                        y_pred=y_pred.cpu().numpy(),
                        y_train=y_train.cpu().numpy(),
                        quantiles=model.config.quantiles,
                        log_scores=False,
                        multioutput='uniform_average'
                        )

            scores_comparison["autoregressive_mean"] = tuple(a + b for a, b in zip(scores_comparison["autoregressive_mean"], (score_mape, score_mase, score_r2, score_wis, score_quantile)))
        else:
            scores_comparison["autoregressive_mean"] = scores_comparison["original"]

    scores_comparison["original"] = tuple(score / len(dataloader_test) for score in scores_comparison["original"])
    scores_comparison["autoregressive_mean"] = tuple(score / len(dataloader_test) for score in scores_comparison["autoregressive_mean"])    

    return scores_comparison, quantiles_comparison, mean_comparison

## Variants

In [11]:
from tirex_loss.loss import QuantileLoss, LossTypes
training_variants = {
    # quantile loss
    'custom_tirex_original_slstm_quantile': {'Autoregressive': False,
                                            'use_slstm': True,
                                            'loss': QuantileLoss,
                                            'quantiles': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
                                            'model_path': Path("../models/custom_tirex_original_slstm_quantile_20260607_135049")
                                            },
    'custom_tirex_original_lstm_quantile': {'Autoregressive': False,
                                            'use_slstm': False,
                                            'loss': QuantileLoss,
                                            'quantiles': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
                                            'model_path': Path("../models/custom_tirex_original_lstm_quantile_20260607_140550")
                                            },
    # 'custom_tirex_autoreg_slstm_quantile': {'Autoregressive': True,
    #                                         'use_slstm': True,
    #                                         'loss': QuantileLoss,
    #                                         'quantiles': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    #                                         },
    # 'custom_tirex_autoreg_lstm_quantile': {'Autoregressive': True,
    #                                         'use_slstm': False,
    #                                         'loss': QuantileLoss,
    #                                         'quantiles': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    #                                         },
    # mse loss
    'custom_tirex_original_slstm_mse': {'Autoregressive': False,
                                            'use_slstm': True,
                                            'loss': LossTypes.MSE.value,
                                            'quantiles': [0.5],
                                            'model_path': Path("../models/custom_tirex_original_slstm_mse_20260628_182306")
                                            },
    'custom_tirex_original_lstm_mse': {'Autoregressive': False,
                                            'use_slstm': False,
                                            'loss': LossTypes.MSE.value,
                                            'quantiles': [0.5],
                                            'model_path': Path("../models/custom_tirex_original_lstm_mse_20260607_144039")
                                            },
    # 'custom_tirex_autoreg_slstm_mse': {'Autoregressive': True,
    #                                         'use_slstm': True,
    #                                         'loss': LossTypes.MSE.value,
    #                                         'quantiles': [0.5]
    #                                         },
    # 'custom_tirex_autoreg_lstm_mse': {'Autoregressive': True,
    #                                         'use_slstm': False,
    #                                         'loss': LossTypes.MSE.value,
    #                                         'quantiles': [0.5]
    #                                         },

    # l1 loss
    'custom_tirex_original_slstm_l1': {'Autoregressive': False,
                                            'use_slstm': True,
                                            'loss': LossTypes.L1.value,
                                            'quantiles': [0.5],
                                            'model_path': Path("../models/custom_tirex_original_slstm_l1_20260614_194203")
                                            },
    'custom_tirex_original_lstm_l1': {'Autoregressive': False,
                                            'use_slstm': False,
                                            'loss': LossTypes.L1.value,
                                            'quantiles': [0.5],
                                            'model_path': Path("../models/custom_tirex_original_lstm_l1_20260607_151456")
                                            },
    # 'custom_tirex_autoreg_slstm_l1': {'Autoregressive': True,
    #                                         'use_slstm': True,
    #                                         'loss': LossTypes.L1.value,
    #                                         'quantiles': [0.5]
    #                                         },
    # 'custom_tirex_autoreg_lstm_l1': {'Autoregressive': True,
    #                                         'use_slstm': False,
    #                                         'loss': LossTypes.L1.value,
    #                                         'quantiles': [0.5]
    #                                         },                                            
}

In [12]:
# debugging tirex not learning on mse with slstm layer
training_variants = {
    # mse loss
    'custom_tirex_original_slstm_mse': {'Autoregressive': False,
                                            'use_slstm': True,
                                            'loss': LossTypes.MSE.value,
                                            'quantiles': [0.5],
                                            'model_path': Path("../models/custom_tirex_original_slstm_mse_20260628_182306")
                                            },
}

In [12]:
context_length = 1024
patch_size = 32

for k, v in training_variants.items():
    quantiles = v['quantiles']
    autoregressive = v['Autoregressive']
    use_slstm = v['use_slstm']
    model_path = v['model_path']
    
    # create a new model
    model = Base_Model(context_length=context_length,
                    quantiles=quantiles,
                    patch_size=patch_size,
                    use_slstm=use_slstm)
    model.to(device)
    
    scores = {}
    quantiles = {}
    for model_file in model_files:
        dict_path = model_path / model_file
        state_dict = torch.load(dict_path, weights_only=True)
        model.load_state_dict(state_dict)

        scores_comparison, quantiles_comparison = compare_autoregressive(dataloader_test, device)
        scores[model_file] = scores_comparison
        quantiles[model_file] = quantiles_comparison

    save_scores(scores, path=model_path / "scores.json")

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

## Pre-trained

In [13]:
# load model
model_path = Path("../models/custom_tirex_original_pretrained_20260614") # just to save scores
model: ForecastModel = load_model("NX-AI/TiRex")
model.to(device)

scores = {}
quantiles = {}
for model_file in model_files:

    scores_comparison, quantiles_comparison, mean_comparison = compare_autoregressive_pretrained(dataloader_test, device)
    scores[model_file] = scores_comparison
    quantiles[model_file] = quantiles_comparison

save_scores(scores, path=model_path / "scores.json")

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

Batches:   0%|          | 0/1261 [00:00<?, ?it/s]

# Summary

## Training

In [17]:
def compute_shared_y_domains_by_loss(training_variants: Dict, padding: float = 1.05) -> Dict:
    """Compute a shared loss y-domain per loss type across training variants.

    Returns a dict mapping loss class -> [0, max_value] domain.
    """
    values_by_loss = {}
    for item in training_variants.values():
        loss_key = item['loss']
        model_path = item['model_path']
        logger = TrainingLogger(log_dir=model_path)
        metrics = logger.load_metrics_file()
        epoch_df = metrics.get('epoch_metrics', None)
        if epoch_df is None:
            continue
        values_by_loss.setdefault(loss_key, [])
        values_by_loss[loss_key].extend(epoch_df['train_loss'].to_list())
        values_by_loss[loss_key].extend(epoch_df['test_loss'].to_list())

    return {
        loss_key: [0, max(values) * padding]
        for loss_key, values in values_by_loss.items()
        if values
    }


# toggle this to switch between shared (per loss type) and independent y-scales
use_shared_y_scale = True

y_domains_by_loss = compute_shared_y_domains_by_loss(training_variants) if use_shared_y_scale else {}

charts = []
summaries = []
for name, item in training_variants.items():
    model_path = item['model_path']
    logger = TrainingLogger(log_dir=model_path)
    metrics = logger.load_metrics_file()
    summary = logger.get_summary()

    y_domain = y_domains_by_loss.get(item['loss']) if use_shared_y_scale else None

    # Tag the chart with a title so each panel is self-labelled
    chart = plot_training_curves(metrics, y_domain=y_domain).properties(
        title=alt.TitleParams(name, fontSize=12, fontWeight="bold"),
        width=280,
        height=180,
    )
    charts.append(chart)
    summaries.append({"model": name, **summary})

In [18]:
# plots
cols = 2
rows = [
    alt.hconcat(*charts[i : i + cols]).resolve_scale(color="independent")
    for i in range(0, len(charts), cols)
]

grid = (
    alt.vconcat(*rows)
    .configure_view(strokeWidth=0)
    .configure_title(anchor="start")
    .properties(title="Training curves – all models")
)
grid

alt.VConcatChart(...)

In [19]:
df = pd.DataFrame(summaries).set_index("model")

# Split model name into architecture + loss for easier scanning
df.index = pd.MultiIndex.from_tuples(
    [tuple(n.rsplit("_", 1)) for n in df.index],
    names=["architecture", "loss"],
)

(
    df.style
    .format({
        "total_epochs": "{:.0f}",
        "best_train_loss": "{:,.2f}",
        "best_test_loss": "{:,.2f}",
        "final_train_loss": "{:,.2f}",
        "final_test_loss": "{:,.2f}",
    })
    .background_gradient(subset=["best_test_loss"], cmap="RdYlGn_r")
    .background_gradient(subset=["best_train_loss"], cmap="RdYlGn_r")
    .set_caption("Model summary – lower loss is better")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th", "props": [("text-align", "left"), ("padding", "6px 12px")]},
        {"selector": "td", "props": [("padding", "5px 12px")]},
    ])
)

,,total_epochs,best_train_loss,best_test_loss,final_train_loss,final_test_loss
architecture,loss,,,,,
custom_tirex_original_slstm,quantile,100,504.15,576.95,506.29,589.72
custom_tirex_original_lstm,quantile,100,383.60,527.99,383.60,540.60
custom_tirex_original_slstm,mse,100,"3,101,287.11","4,775,838.26","3,119,640.25","4,912,296.42"
custom_tirex_original_lstm,mse,100,"1,293,486.37","3,125,015.24","1,293,486.37","3,136,792.49"
custom_tirex_original_slstm,l1,100,"1,509.92","1,519.90","1,525.06","1,591.28"
custom_tirex_original_lstm,l1,100,974.00,"1,384.28",975.28,"1,394.63"


## Evaluation

In [20]:
model_paths = {}
for name, item in training_variants.items():
    model_path = item['model_path']
    model_paths[name] = model_path
model_paths['original_pretrained'] = Path("../models/tirex_original_pretrained_20260614")
model_paths

{'custom_tirex_original_slstm_quantile': WindowsPath('../models/custom_tirex_original_slstm_quantile_20260607_135049'),
 'custom_tirex_original_lstm_quantile': WindowsPath('../models/custom_tirex_original_lstm_quantile_20260607_140550'),
 'custom_tirex_original_slstm_mse': WindowsPath('../models/custom_tirex_original_slstm_mse_20260628_182306'),
 'custom_tirex_original_lstm_mse': WindowsPath('../models/custom_tirex_original_lstm_mse_20260607_144039'),
 'custom_tirex_original_slstm_l1': WindowsPath('../models/custom_tirex_original_slstm_l1_20260614_194203'),
 'custom_tirex_original_lstm_l1': WindowsPath('../models/custom_tirex_original_lstm_l1_20260607_151456'),
 'original_pretrained': WindowsPath('../models/tirex_original_pretrained_20260614')}

In [ ]:
scores = []
for name, p in model_paths.items():
    with open(p / "scores.json", 'r') as f:
        s = json.load(f)
    scores.append({"model": name, **s})

SCORE_COLS = ["score_smape", "score_mase", "score_r2", "score_wis", "score_quantile"]

rows = []
for entry in scores:          # your list of dicts
    model = entry["model"]
    best = entry["best_model.pth"]
    for pred_mode, values in best.items():
        rows.append({
            "model": model,
            "pred_mode": pred_mode,
            **dict(zip(SCORE_COLS, values)),
        })

df_scores = pd.DataFrame(rows).set_index(["model", "pred_mode"])

# Build gradient styles — lower=better for smape/mase/wis, higher=better for r2
lower_better = ["score_smape", "score_mase", "score_wis", "score_quantile"]
higher_better = ["score_r2"]

(
    df_scores.style
    .format({
        "score_smape": "{:.4f}",
        "score_mase":  "{:.4f}",
        "score_r2":    "{:.4f}",
        "score_wis":   "{:,.1f}",
    })
    .background_gradient(subset=lower_better,  cmap="RdYlGn_r")  # red=bad, green=good
    .background_gradient(subset=higher_better, cmap="RdYlGn")    # green=good (no _r)
    .set_caption("Best model scores by model and prediction mode")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold"), ("text-align", "left"), ("padding-bottom", "8px")]},
        {"selector": "th", "props": [("text-align", "left"), ("padding", "6px 12px"), ("white-space", "nowrap")]},
        {"selector": "td", "props": [("padding", "5px 12px"), ("white-space", "nowrap")]},
    ]) 
)    